# 01 - De-duplicate, then train/test split (v2)

Goal: remove exact duplicate rows, then split the remaining rows **once**, before any preprocessing, and save which row went where.

- Exact duplicates (all 15 columns) are dropped first, keeping the first copy. Otherwise a row could sit in train while its identical copy sits in test, which leaks training rows into the test.
- The dropped rows are recorded in `results/dropped_duplicates.csv`.
- Stratify on income x sex so both parts keep the same class balance and the same sex mix.
- 80/20 split, `random_state=42`, so the split is reproducible.
- No preprocessing here. The raw file is only read, never written. `row_id` always means the 0-based row position in `data/adult.csv`.
- v1 (split without de-duplication) is archived in `notebooks/archive_v1/` and `results/archive_v1/`.

## 1. Kernel and library versions
Record the versions so results can be reproduced later. `sys.executable` should be `C:\Users\User\venvs\classification\Scripts\python.exe` (kernel "Python (classification)"). If not, stop and switch the kernel.

In [11]:
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

print('Python executable:', sys.executable)
print('pandas      ', pd.__version__)
print('numpy       ', np.__version__)
print('scikit-learn', sklearn.__version__)
print('matplotlib  ', matplotlib.__version__)

Python executable: c:\Users\User\venvs\classification\Scripts\python.exe
pandas       3.0.6
numpy        2.4.6
scikit-learn 1.9.1
matplotlib   3.11.2


## 2. Paths
The notebook may start in `notebooks/` or in the project root, depending on how Cursor launches it.
We look for `data/adult.csv` in the current folder, then in its parent.

In [12]:
ROOT = Path.cwd()
if not (ROOT / 'data' / 'adult.csv').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'adult.csv'
RESULTS_PATH = ROOT / 'results'

assert DATA_PATH.exists(), f'Raw data not found at {DATA_PATH}'
print('Project root:', ROOT)
print('Data file:   ', DATA_PATH)

Project root: c:\Users\User\OneDrive\Desktop\Classification_Model
Data file:    c:\Users\User\OneDrive\Desktop\Classification_Model\data\adult.csv


## 3. Load the raw data (read-only)
- `header=None`: the file has no header row.
- `names=...`: the 15 column names from CLAUDE.md, in file order.
- `keep_default_na=False`: keep `"?"` and any other text exactly as it is in the file; pandas does not convert anything to NaN on its own.

In [13]:
COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income',
]

df = pd.read_csv(DATA_PATH, header=None, names=COLUMNS, keep_default_na=False)

print('Shape:', df.shape)
assert df.shape == (48842, 15), f'Unexpected shape {df.shape}'

Shape: (48842, 15)


## 4. Drop exact duplicates (keep the first copy)
- A row is a duplicate if **all 15 columns** equal those of an earlier row. The earliest copy is kept; every later copy is dropped.
- We expect exactly **52** dropped rows and **48,790** kept rows. If the numbers differ, the notebook stops: the data or the definition of a duplicate must be checked first.
- `df` itself is not changed. We only build the list of **kept row positions**; `row_id` keeps meaning the position in `data/adult.csv`.
- `results/dropped_duplicates.csv` records each dropped `row_id` and the `kept_row_id` of its kept copy.
- Saving: if a results file already exists, it is **not overwritten**. We load it and assert it is identical (the same guard as in 02). To regenerate on purpose, move the old file to an archive folder first.

In [14]:
EXPECTED_DROPPED = 52
EXPECTED_KEPT = 48790


def save_or_verify(frame, path):
    """Save frame as CSV; if the file already exists, assert it is identical instead."""
    if path.exists():
        existing = pd.read_csv(path)
        try:
            pd.testing.assert_frame_equal(existing, frame, check_dtype=False)
        except AssertionError as err:
            raise AssertionError(
                f'{path} already exists and differs from what was just computed. '
                'Not overwriting. Move the old file to an archive folder if a new version is intended.'
            ) from err
        print('Already exists and is identical; not overwritten:', path)
    else:
        path.parent.mkdir(exist_ok=True)
        frame.to_csv(path, index=False)
        print('Saved:', path)


positions = np.arange(len(df))
dup_mask = df.duplicated(subset=COLUMNS, keep='first').to_numpy()
n_dropped = int(dup_mask.sum())
n_kept = len(df) - n_dropped
print('Exact duplicate rows to drop:', n_dropped)
print('Rows kept:                   ', n_kept)

assert n_dropped == EXPECTED_DROPPED, (
    f'STOP: expected {EXPECTED_DROPPED} duplicate rows to drop, found {n_dropped}. '
    'Check the raw file and the duplicate definition before continuing.'
)
assert n_kept == EXPECTED_KEPT, (
    f'STOP: expected {EXPECTED_KEPT} kept rows, found {n_kept}.'
)

# For every row: the position of the first row with identical values in all 15 columns
first_pos = (
    df.assign(_pos=positions)
    .groupby(COLUMNS, sort=False)['_pos']
    .transform('min')
    .to_numpy()
)
assert (first_pos[~dup_mask] == positions[~dup_mask]).all(), 'A kept row does not point to itself'

dropped_df = pd.DataFrame({
    'row_id': positions[dup_mask],
    'kept_row_id': first_pos[dup_mask],
}).sort_values('row_id').reset_index(drop=True)

assert (dropped_df['kept_row_id'] < dropped_df['row_id']).all(), 'A kept copy is not earlier than its dropped copy'
assert not dup_mask[dropped_df['kept_row_id'].to_numpy()].any(), 'A kept_row_id points to a dropped row'

kept_positions = positions[~dup_mask]
assert len(kept_positions) == EXPECTED_KEPT

print()
print('First dropped rows:')
print(dropped_df.head().to_string(index=False))
print()
save_or_verify(dropped_df, RESULTS_PATH / 'dropped_duplicates.csv')
print('Rows:', len(dropped_df))

Exact duplicate rows to drop: 52
Rows kept:                    48790

First dropped rows:
 row_id  kept_row_id
   4881         4325
   5104         2303
   9171         4767
  11631         5805
  13084         5842

Already exists and is identical; not overwritten: c:\Users\User\OneDrive\Desktop\Classification_Model\results\dropped_duplicates.csv
Rows: 52


## 5. Stratification label: income x sex (kept rows only)
We first check that `income` and `sex` hold only the expected values. A stray space or a trailing `.` would otherwise quietly create extra groups.
Then we combine them into four groups, e.g. `>50K_Female`. From here on only the **kept** rows are used. The label's index is the original row position.

In [15]:
kept_df = df.iloc[kept_positions]

income_values = set(kept_df['income'].unique())
sex_values = set(kept_df['sex'].unique())
assert income_values == {'<=50K', '>50K'}, f'Unexpected income values: {income_values}'
assert sex_values == {'Male', 'Female'}, f'Unexpected sex values: {sex_values}'

strat_kept = kept_df['income'] + '_' + kept_df['sex']
print(strat_kept.value_counts())

<=50K_Male      22702
<=50K_Female    14407
>50K_Male        9912
>50K_Female      1769
Name: count, dtype: int64


## 6. Split the kept row positions
We split the **kept row positions** (original positions in `data/adult.csv`), not the data itself. That keeps the split a small, reusable list of row numbers.
`test_size=0.2`, `random_state=42`, stratified on the four-group label.

In [16]:
train_idx, test_idx = train_test_split(
    kept_positions,
    test_size=0.2,
    random_state=42,
    stratify=strat_kept,
)

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

## 7. Sizes and shares
If stratification worked, the >50K share and the four-group shares should be almost identical overall (all kept rows), in train and in test.

In [17]:
print(f'Kept rows:  {len(kept_positions):,}')
print(f'Train size: {len(train_idx):,}')
print(f'Test size:  {len(test_idx):,}')
print()

for name, part in [('Overall', kept_df), ('Train', train_df), ('Test', test_df)]:
    share = (part['income'] == '>50K').mean()
    print(f'>50K share {name:<8}: {share:.4%}')
print()

group_shares = pd.DataFrame({
    'overall': strat_kept.value_counts(normalize=True),
    'train': strat_kept.loc[train_idx].value_counts(normalize=True),
    'test': strat_kept.loc[test_idx].value_counts(normalize=True),
})
print('Four-group shares:')
print((group_shares * 100).round(3).astype(str) + ' %')

Kept rows:  48,790
Train size: 39,032
Test size:  9,758

>50K share Overall : 23.9414%
>50K share Train   : 23.9419%
>50K share Test    : 23.9393%

Four-group shares:
               overall     train      test
<=50K_Male     46.53 %  46.528 %  46.536 %
<=50K_Female  29.529 %   29.53 %  29.524 %
>50K_Male     20.316 %  20.317 %  20.312 %
>50K_Female    3.626 %   3.625 %   3.628 %


## 8. Sanity checks
- No row number may appear in both parts (that would be direct leakage).
- Together the two parts must cover every **kept** row exactly once, and no dropped duplicate may appear in either part.
- `fnlwgt` must be present in both parts, because we use it later as a sample weight.

In [18]:
overlap = set(train_idx) & set(test_idx)
assert len(overlap) == 0, f'{len(overlap)} row numbers are in both parts'

assert len(train_idx) + len(test_idx) == len(kept_positions), 'Parts do not add up to all kept rows'
assert set(train_idx) | set(test_idx) == set(kept_positions), 'Split does not match the kept rows exactly'

dropped_in_split = set(dropped_df['row_id']) & (set(train_idx) | set(test_idx))
assert len(dropped_in_split) == 0, f'{len(dropped_in_split)} dropped duplicates are in the split'

for name, part in [('train', train_df), ('test', test_df)]:
    assert 'fnlwgt' in part.columns, f'fnlwgt missing from {name}'
    assert part['fnlwgt'].notna().all(), f'fnlwgt has missing values in {name}'

print('All checks passed.')

All checks passed.


## 9. No exact duplicates across the split
In v1, 17 test rows had an exact copy (all 15 columns) in train. After de-duplication this must be **0**: no kept row is duplicated anywhere, and in particular no test row has a copy in train.

In [19]:
assert not kept_df.duplicated().any(), 'Kept rows still contain exact duplicates'

matched = test_df.merge(train_df.drop_duplicates(), on=COLUMNS, how='left', indicator=True)
n_test_rows_with_copy_in_train = int((matched['_merge'] == 'both').sum())

print('Test rows with an exact copy in train:', n_test_rows_with_copy_in_train)
assert n_test_rows_with_copy_in_train == 0, 'Duplicates cross the split after de-duplication'

Test rows with an exact copy in train: 0


## 10. Save the split
We save only row numbers and the split label, not the data. Later notebooks load the raw file and this list, so everyone uses the same split.
`row_id` is the 0-based position in `data/adult.csv` (row 0 = the first line of the file). Only kept rows are listed; dropped duplicates are in `dropped_duplicates.csv`.
If `split_indices.csv` already exists it is not overwritten; it must be identical (same guard as above).

In [20]:
split_df = pd.DataFrame({
    'row_id': np.concatenate([train_idx, test_idx]),
    'split': ['train'] * len(train_idx) + ['test'] * len(test_idx),
}).sort_values('row_id').reset_index(drop=True)

save_or_verify(split_df, RESULTS_PATH / 'split_indices.csv')
print('Rows: ', len(split_df))
print(split_df['split'].value_counts())

Already exists and is identical; not overwritten: c:\Users\User\OneDrive\Desktop\Classification_Model\results\split_indices.csv
Rows:  48790
split
train    39032
test      9758
Name: count, dtype: int64
